## Precompute embeddings for the full CUAD dataset (run this first)

Embedding the documents locally is the slow step in a full run (measured ~2.8s/sample, and it's
the bottleneck now that generation moved to a fast per-token API). This notebook embeds and
**persists** a Chroma vector store for every CUAD sample **once**, so later experiment runs hit the
cache (`get_or_build_vector_store` returns `cached=True`) and skip re-embedding entirely.

**Reuse contract — the run notebooks must match all three of these or the cache won't hit:**
1. Same **embedder** (`nomic-embed-text-v2-moe`)
2. Same **chunking** (`chunk_size=500`, `chunk_overlap=50`)
3. Same **`COLLECTION_TAG`** (`nomic_cs500_co50`) and same **`CHROMA_PERSIST_DIR`**

The tag encodes the embedder + chunk config on purpose: `get_or_build_vector_store` keys the cache
on the collection name only (not the embedder), so changing embedder/chunking **must** change the
tag — otherwise a run would load stale vectors. Keep the tag in sync across `00`, `05`, etc.

**Resumable:** thanks to the cache fix, re-running this notebook after an interruption skips
samples already embedded and only builds the missing ones.

### Prerequisites

- Local Ollama running with the embedder pulled: `ollama pull nomic-embed-text-v2-moe:latest`.
- `CHROMA_PERSIST_DIR` set in `.env` to a stable local path (the run notebooks must use the same).

In [5]:
import os
import time

from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.env import load_secrets
from reliablerag.providers import create_embeddings
from reliablerag.retriever import get_or_build_vector_store

### Configuration

`COLLECTION_TAG` and `CHROMA_PERSIST_DIR` here are the shared cache key — the run notebooks must
use identical values to reuse these embeddings.

In [6]:
load_secrets()

EMBEDDING_MODEL    = "nomic-embed-text-v2-moe:latest"   # local Ollama
CHUNK_SIZE         = 500
CHUNK_OVERLAP      = 50
COLLECTION_TAG     = f"nomic_cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"   # shared cache key — keep in sync with run notebooks
CHROMA_PERSIST_DIR = os.environ.get("CHROMA_PERSIST_DIR", "./data/chroma_db")

N_SAMPLES = None   # None = full dataset; set an int to embed only the first N

print(f"Embedder      : {EMBEDDING_MODEL}")
print(f"Chunking      : size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Collection tag: {COLLECTION_TAG}")
print(f"Persist dir   : {CHROMA_PERSIST_DIR}")

Embedder      : nomic-embed-text-v2-moe:latest
Chunking      : size=500, overlap=50
Collection tag: nomic_cs500_co50
Persist dir   : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [7]:
embeddings = create_embeddings("ollama", EMBEDDING_MODEL)

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset) if N_SAMPLES is None else list(dataset.select(range(N_SAMPLES)))
print(f"Loaded {len(samples)} CUAD samples")

Loaded 1530 CUAD samples


### Build & persist a vector store per sample

Uses the exact chunking + collection naming (`cuad_{i}_{tag}`) that `run_rag_experiment` uses, so
the collections these produce are the ones the runs will look up.

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

built, cached, total_chunks = 0, 0, 0
t_start = time.perf_counter()

for i, sample in enumerate(samples):
    raw_doc   = sample["documents"][0]
    coll_name = f"cuad_{i}_{COLLECTION_TAG}"
    chunks    = splitter.create_documents([raw_doc], metadatas=[{"source": f"cuad_sample_{i}"}])

    _, is_cached = get_or_build_vector_store(
        chunks, embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    if is_cached:
        cached += 1
    else:
        built += 1
        total_chunks += len(chunks)

    if (i + 1) % 25 == 0 or (i + 1) == len(samples):
        elapsed = time.perf_counter() - t_start
        print(f"[{i+1}/{len(samples)}] built={built} cached={cached} "
              f"chunks_embedded={total_chunks} elapsed={elapsed:.1f}s")

print(f"\nDone — {built} built, {cached} already cached, {total_chunks} chunks embedded, "
      f"{time.perf_counter() - t_start:.1f}s total")

[25/1530] built=0 cached=25 chunks_embedded=0 elapsed=0.1s
[50/1530] built=0 cached=50 chunks_embedded=0 elapsed=0.2s
[75/1530] built=0 cached=75 chunks_embedded=0 elapsed=0.3s
[100/1530] built=0 cached=100 chunks_embedded=0 elapsed=0.4s
[125/1530] built=0 cached=125 chunks_embedded=0 elapsed=0.5s
[150/1530] built=0 cached=150 chunks_embedded=0 elapsed=0.6s
[175/1530] built=0 cached=175 chunks_embedded=0 elapsed=0.7s
[200/1530] built=0 cached=200 chunks_embedded=0 elapsed=0.8s
[225/1530] built=0 cached=225 chunks_embedded=0 elapsed=1.0s
[250/1530] built=0 cached=250 chunks_embedded=0 elapsed=1.1s
[275/1530] built=0 cached=275 chunks_embedded=0 elapsed=1.2s
[300/1530] built=0 cached=300 chunks_embedded=0 elapsed=1.4s
[325/1530] built=0 cached=325 chunks_embedded=0 elapsed=1.5s
[350/1530] built=0 cached=350 chunks_embedded=0 elapsed=1.6s
[375/1530] built=0 cached=375 chunks_embedded=0 elapsed=1.7s
[400/1530] built=0 cached=400 chunks_embedded=0 elapsed=1.8s
[425/1530] built=0 cached=425 

### Verify

Re-running the build cell now should report **every** sample as `cached` (0 built) — confirming the
persisted collections are reusable. The run notebooks (with matching embedder / chunking /
`COLLECTION_TAG` / `CHROMA_PERSIST_DIR`) will then skip embedding and go straight to retrieval.